# CheckThat 2025 — Reasoning Trace Generation (Qwen3-8B, HF pipeline, 2xT4)

Goal: per (claim, evidences) pair, gen 20 reasoning traces at 20 diff fixed temps.
Few-shot prompted. Output label ∈ {True, False, Conflicting}.
This run: first 5 claims only, for manual QA. Scale later by removing slice.


In [1]:
# Env check + installs
# Root cause of `StrictDataclassDefinitionError: Qwen3Config must be a dataclass before
# applying @strict`: Kaggle's pre-baked transformers/huggingface_hub versions are mismatched
# (Qwen3Config's @strict decorator needs a transformers version released together with a
# compatible huggingface_hub). Fix: upgrade BOTH together to their current latest release.
!pip install -q -U "transformers>=4.57" "huggingface_hub>=0.35" "tokenizers>=0.22" "accelerate>=1.11" "bitsandbytes>=0.48"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 50.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 

**IMPORTANT — restart required.** Kaggle's kernel process already imported the old
`transformers`/`huggingface_hub` into memory before this install ran. `pip install -U`
does not hot-swap already-loaded modules. After running the cell above:

1. Click **Run > Restart session** (or the restart icon in the Kaggle toolbar).
2. Then **Run All** again from the top (the pip install will just say "already satisfied"
   and skip straight to using the new versions).

Skipping the restart is the #1 cause of this exact error persisting after "fixing" it.


In [2]:
# sanity check versions after restart
import transformers, huggingface_hub, tokenizers, accelerate
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)
print("accelerate:", accelerate.__version__)

# quick import test — this line alone reproduces the bug if versions are still stale
from transformers import Qwen3Config
print("Qwen3Config imported OK")


transformers: 5.16.1
huggingface_hub: 1.29.0
tokenizers: 0.23.1
accelerate: 1.14.0
Qwen3Config imported OK


In [3]:
import json, os, re, time, gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen3-8B"
DEVICE_MAP = "auto"  # spread across 2x T4 (16GB each = 32GB total)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())


CUDA available: True
GPU count: 2


## Load model

Qwen3-8B is ungated — no HF token needed. Instruct-tuned, has chat template + built-in
"thinking mode" (emits `<think>...</think>` block before answer). We disable thinking mode
(`enable_thinking=False`) so output is a direct justification matching the few-shot style,
not raw exposed CoT. 4-bit NF4 quant via bitsandbytes for 2xT4 headroom.


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map=DEVICE_MAP,
)
model.eval()
print("Model loaded. Device map:", model.hf_device_map)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded. Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## Few-shot prompt template

System + few-shot examples (from CheckThat annotated samples) + target claim/evidence.
Model must output free-text justification ending in `Label: True|False|Conflicting`.


In [5]:
SYSTEM_PROMPT = (
    "You are a numerical fact-checking assistant for the CLEF CheckThat! 2025 claim verification task. "
    "Given a CLAIM and a set of EVIDENCE snippets, write a short justification reasoning about "
    "whether the evidence supports, contradicts, or gives conflicting signals about the claim. "
    "End your answer with a single line in the exact format: 'Label: True' or 'Label: False' or "
    "'Label: Conflicting'. Do not add anything after the Label line."
)

FEWSHOT_EXAMPLES = [
    {
        "claim": "Six masks worn to school by children were sent to a University of Florida laboratory and found to carry at least 11 dangerous pathogens.",
        "evidences": [
            "15 juin 2021  the resulting report found that five masks were contaminated with bacteria, parasites, and fungi, including three with dangerous pathogenic and ...",
            "in june 2021, a research facility at the university of florida tested six masks worn by children during their school day and found that at least 11 disease-causing pathogens were present on the mask material.",
            "gainesville parents in florida concerned about the harm caused to their children wearing face masks all day at school in 90 f weather sent out six masksfive that were worn by children ages 6 to 11 for five to eight hours at school, and one worn by an adultto be analyzed for contaminants at the university of floridas mass spectrometry research and education center. of the six masks, three were surgical, two cotton, and a poly gaiter. masks that have not been worn and a t-shirt worn at school acted as the control samples. a laboratory at the university of florida that recently analyzed a small sample of face masks, detected the presence of 11 dangerous pathogens that included bacterias that cause diphtheria, pneumonia, and meningitis."
        ],
        "reasoning_trace": (
            "Justification: The claim states that six masks worn by children were tested and found to "
            "carry at least 11 dangerous pathogens. The evidence confirms this, stating that a research "
            "facility at the University of Florida tested six masks worn by children and found that there "
            "were indeed 11 disease-causing pathogens present. Therefore, the numerical span in the claim "
            "matches with the evidence provided.\n\nLabel: True"
        ),
    },
    {
        "claim": "Six masks worn to school by children were sent to a University of Florida laboratory and found to carry at least 11 dangerous pathogens.",
        "evidences": [
            "15 juin 2021  the resulting report found that five masks were contaminated with bacteria, parasites, and fungi, including three with dangerous pathogenic and ...",
            "in june 2021, a research facility at the university of florida tested six masks worn by children during their school day and found that at least 11 disease-causing pathogens were present on the mask material.",
            "gainesville parents in florida concerned about the harm caused to their children wearing face masks all day at school in 90 f weather sent out six masksfive that were worn by children ages 6 to 11 for five to eight hours at school, and one worn by an adultto be analyzed for contaminants at the university of floridas mass spectrometry research and education center. of the six masks, three were surgical, two cotton, and a poly gaiter. masks that have not been worn and a t-shirt worn at school acted as the control samples. a laboratory at the university of florida that recently analyzed a small sample of face masks, detected the presence of 11 dangerous pathogens that included bacterias that cause diphtheria, pneumonia, and meningitis."
        ],
        "reasoning_trace": (
            "Justification: The evidence clearly supports the claim that masks worn by children were "
            "tested and found to carry at least 11 dangerous pathogens. It mentions that six masks were "
            "analyzed, confirming the presence of these pathogens as stated in the claim.\n\nLabel: True"
        ),
    },
]

def build_messages(claim, evidences):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEWSHOT_EXAMPLES:
        user_turn = "CLAIM: " + ex["claim"] + "\n\nEVIDENCE:\n" + "\n".join(
            f"- {e}" for e in ex["evidences"]
        )
        msgs.append({"role": "user", "content": user_turn})
        msgs.append({"role": "assistant", "content": ex["reasoning_trace"]})
    target_turn = "CLAIM: " + claim + "\n\nEVIDENCE:\n" + "\n".join(f"- {e}" for e in evidences)
    msgs.append({"role": "user", "content": target_turn})
    return msgs


## Temperature schedule (20 fixed values, 0.0 → 1.0)

Even spread across [0.0, 1.0], 20 steps. temp=0.0 -> greedy (near-deterministic) baseline trace,
rest ramp up sampling diversity linearly.


In [6]:
import numpy as np

TEMPERATURES = [float(t) for t in np.round(np.linspace(0.0, 1.0, 20), 3)]
assert len(TEMPERATURES) == 20
print(TEMPERATURES)


[0.0, 0.053, 0.105, 0.158, 0.211, 0.263, 0.316, 0.368, 0.421, 0.474, 0.526, 0.579, 0.632, 0.684, 0.737, 0.789, 0.842, 0.895, 0.947, 1.0]


## Label extraction + generation fn


In [7]:
LABEL_RE = re.compile(r"Label:\s*(True|False|Conflicting)", re.IGNORECASE)

def extract_label(text):
    m = LABEL_RE.search(text)
    if m:
        return m.group(1).capitalize() if m.group(1).lower() != "conflicting" else "Conflicting"
    return None

def normalize_label(raw):
    if raw is None:
        return None
    r = raw.strip().lower()
    if r == "true":
        return "True"
    if r == "false":
        return "False"
    if r.startswith("conflict"):
        return "Conflicting"
    return None

@torch.inference_mode()
def generate_trace(claim, evidences, temperature, max_new_tokens=2500, seed=None):
    messages = build_messages(claim, evidences)
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=True,  # Qwen3: skip <think> block, direct justification only
        return_tensors="pt",
        return_dict=True,  # returns BatchEncoding {input_ids, attention_mask}, not a bare tensor
    ).to(model.device)

    if seed is not None:
        torch.manual_seed(seed)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=max(temperature, 0.01),
        top_p=0.95,
        top_k=20,
        pad_token_id=tokenizer.pad_token_id,
    )
    out = model.generate(**inputs, **gen_kwargs)
    input_len = inputs["input_ids"].shape[-1]
    new_tokens = out[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    label = normalize_label(extract_label(text))
    return text, label


## Input data

Load input JSON in the agreed schema:
```json
[{"claim": "...", "original_id": 1, "number_of_unique_docs": 3, "docs": ["...", "...", "..."]}, ...]
```
Adjust `INPUT_PATH` to your Kaggle dataset mount.


In [8]:
INPUT_PATH = "/kaggle/input/datasets/d202511054/numerical-fact-checking-dataset-dataset/3-processed-reranked.json"  # <-- adjust to your dataset path

# Current run output
OUTPUT_PATH = "/kaggle/working/reasoning_traces.json"

# Previously generated reasoning traces.
# Upload the previous JSON to the Kaggle notebook/session and set this path.
# Example:
# PREVIOUS_TRACE_PATH = "/kaggle/input/my-previous-traces/reasoning_traces_first5.json"
PREVIOUS_TRACE_PATH = "/kaggle/input/datasets/d202511054/clef-2025-test-reasoning-traces/reasoning_traces.json"  # <-- adjust

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    all_claims = json.load(f)

print(f"Total claims in input: {len(all_claims)}")

# Process all claims; already-processed claims are skipped below.
SUBSET = all_claims
print(f"Claims available for processing: {len(SUBSET)}")


Total claims in input: 3124
Claims available for processing: 3124


## Resume-safe reasoning-trace generation\n
\n
Claims found in either `PREVIOUS_TRACE_PATH` or `OUTPUT_PATH` are skipped by `original_id`.\n
The previous results are also carried forward into the new output file.\n

In [ ]:
# ============================================================
# LOAD PREVIOUS/CURRENT RESULTS AND BUILD THE SKIP SET
# ============================================================

def load_trace_results(path):
    """Load a reasoning-trace JSON file if it exists."""
    if not path:
        return []

    if not os.path.exists(path):
        print(f"No previous trace file found at: {path}")
        return []

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"Expected a JSON list in {path}, got {type(data).__name__}"
        )

    # Keep only valid claim records with an original_id.
    valid = [
        r for r in data
        if isinstance(r, dict) and "original_id" in r
    ]

    print(f"Loaded {len(valid)} processed claims from: {path}")
    return valid


# Start with traces from the separately supplied previous JSON.
results = load_trace_results(PREVIOUS_TRACE_PATH)

# Also resume from the current output file, if it already exists.
# Current output entries take precedence if the same original_id appears
# in both files.
current_results = load_trace_results(OUTPUT_PATH)

by_id = {}

for record in results:
    by_id[record["original_id"]] = record

for record in current_results:
    by_id[record["original_id"]] = record

results = list(by_id.values())

# IMPORTANT:
# A claim is considered already processed if its original_id occurs in
# either the previous trace file OR the current output file.
done_ids = {record["original_id"] for record in results}

print(f"Previously/currently processed unique claims: {len(done_ids)}")
print(f"New claims to process: {sum(item['original_id'] not in done_ids for item in SUBSET)}")


# ============================================================
# GENERATE TRACES ONLY FOR CLAIMS NOT IN done_ids
# ============================================================

for item in SUBSET:
    claim_id = item["original_id"]

    if claim_id in done_ids:
        print(f"Skip {claim_id} (already processed in previous/current trace JSON)")
        continue

    claim = item["claim"]
    docs = item["docs"]
    print(f"\n=== Claim {claim_id}: {claim[:80]}... ===")

    traces = []
    for i, temp in enumerate(TEMPERATURES, start=1):
        t0 = time.time()
        text, label = generate_trace(
            claim,
            docs,
            temperature=temp,
            seed=1000 * claim_id + i
        )
        dt = time.time() - t0

        traces.append({
            "trace_id": i,
            "temperature": temp,
            "label": label,
            "reasoning_trace": text,
        })

        print(
            f"  trace {i:2d}  T={temp:.2f}  "
            f"label={label}  ({dt:.1f}s)"
        )

    new_record = {
        "claim": claim,
        "original_id": claim_id,
        "reasoning_traces": traces,
    }

    results.append(new_record)

    # Mark immediately so the same ID can never be generated twice
    # during this run.
    done_ids.add(claim_id)

    # Incremental checkpoint after every claim.
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    gc.collect()
    torch.cuda.empty_cache()

print("\nDone. Saved to", OUTPUT_PATH)
print(f"Total processed records in output: {len(results)}")


Loaded 44 processed claims from: /kaggle/input/datasets/d202511054/clef-2025-test-reasoning-traces/reasoning_traces.json
No previous trace file found at: /kaggle/working/reasoning_traces.json
Previously/currently processed unique claims: 44
New claims to process: 3080
Skip 1 (already processed in previous/current trace JSON)
Skip 2 (already processed in previous/current trace JSON)
Skip 3 (already processed in previous/current trace JSON)
Skip 4 (already processed in previous/current trace JSON)
Skip 5 (already processed in previous/current trace JSON)
Skip 6 (already processed in previous/current trace JSON)
Skip 7 (already processed in previous/current trace JSON)
Skip 8 (already processed in previous/current trace JSON)
Skip 9 (already processed in previous/current trace JSON)
Skip 10 (already processed in previous/current trace JSON)
Skip 11 (already processed in previous/current trace JSON)
Skip 12 (already processed in previous/current trace JSON)
Skip 13 (already processed in pr

## Quick QA view


In [ ]:
for r in results:
    print("="*100)
    print("Claim:", r["claim"])
    print("ID:", r["original_id"])
    label_counts = {}
    for t in r["reasoning_traces"]:
        label_counts[t["label"]] = label_counts.get(t["label"], 0) + 1
    print("Label distribution across 20 traces:", label_counts)


In [ ]:
print(0)